# 第三章：立体视觉点云重建

## 编程实践：手写 SFM 点云重建（F/E/R,t/三角化）

| 项目 | 说明 |
|------|------|
| 输入图片 | `stereo_image_1.jpg` / `stereo_image_2.jpg`（参考库第 19 章为外链 SFM 数据集，本仓库沿用自备立体对） |
| 手写核心 | 归一化 8 点法求 F、E 分解恢复 R/t、线性三角化、点云导出 |
| 允许调用 | 图像读写、矩阵计算、特征点提取与匹配（OpenCV/numpy） |
| 对比验证 | 与 OpenCV `cv2.findFundamentalMat` 求出的 F 做数值对比 |


## 一、学习目标

1. 掌握从图像序列重建三维点云的计算原理与编程实现。
2. 掌握**对极几何**：基础矩阵 `F` 与本质矩阵 `E` 的关系。
3. 掌握 **R/t 恢复** 与 **线性三角化** 的完整 SFM 流程。

### 对极几何

```
x2^T F x1 = 0         （像素坐标）
E = K2^T F K1         （归一化坐标）
```

`F` 有 7 个自由度，最少 8 对匹配点求解（8 点法）；`E` 与相机内参有关。

### E 分解与三角化

对 `E` 做 SVD，可得到 4 组候选 `(R, t)`；用 **cheirality（正深度）** 准则选择正确解：

```
P1 = K [I | 0]
P2 = K [R | t]
```

对每对匹配点，用 DLT 构造 4x4 线性方程组并 SVD 求解三维点 `X`。


## 二、手写约束清单（二阶段）

- ✅ 允许调用：`cv_imread` / `cv_imwrite`（读写）、numpy 矩阵计算（SVD、norm 等）、`cv2.SIFT_create` / `cv2.BFMatcher`（特征提取与匹配）。
- ❌ 其余必须手写：8 点法求 F、E 分解、cheirality 选择、线性三角化、点云导出与可视化。
- 相机内参 `K` 采用 `[[600,0,250],[0,600,200],[0,0,1]]`（与图片尺寸 500×400 对应的假设内参，参考库 SFM 数据集未附标定参数）。


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
import math


def normalize_points(pts):
    """Hartley 归一化：平移质心到原点，缩放到平均距离 sqrt(2)。"""
    pts = np.asarray(pts, dtype=np.float64)
    c = pts.mean(axis=0)
    d = np.mean(np.linalg.norm(pts - c, axis=1))
    s = math.sqrt(2.0) / d
    return np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])


def compute_F_8point(pts1, pts2):
    """手写归一化 8 点法求基础矩阵 F，并强制 rank(F)=2。"""
    T1 = normalize_points(pts1)
    T2 = normalize_points(pts2)
    h1 = (T1 @ np.hstack([pts1, np.ones((len(pts1), 1))]).T).T
    h2 = (T2 @ np.hstack([pts2, np.ones((len(pts2), 1))]).T).T

    A = []
    for (x1, y1, _), (x2, y2, _) in zip(h1, h2):
        A.append([x1 * x2, x1 * y2, x1, y1 * x2, y1 * y2, y1, x2, y2, 1.0])
    A = np.array(A)
    _, _, Vt = np.linalg.svd(A)
    F = Vt[-1].reshape(3, 3)

    U, S, Vt = np.linalg.svd(F)
    S[-1] = 0.0
    F = U @ np.diag(S) @ Vt
    F = T2.T @ F @ T1
    return F / F[2, 2]


def compute_E(K1, K2, F):
    """E = K2^T F K1。"""
    return K2.T @ F @ K1


def decompose_E(E):
    """对本质矩阵 SVD 分解，返回 4 组候选 (R, t)。"""
    U, _, Vt = np.linalg.svd(E)
    W = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
    R1 = U @ W @ Vt
    R2 = U @ W.T @ Vt
    if np.linalg.det(R1) < 0:
        R1 = -R1
    if np.linalg.det(R2) < 0:
        R2 = -R2
    t = U[:, 2]
    return [(R1, t), (R1, -t), (R2, t), (R2, -t)]


def triangulate_linear(P1, P2, pts1, pts2):
    """手写线性三角化：DLT + SVD 求解三维点。"""
    points = []
    for (x1, y1), (x2, y2) in zip(pts1, pts2):
        A = np.zeros((4, 4))
        A[0] = x1 * P1[2] - P1[0]
        A[1] = y1 * P1[2] - P1[1]
        A[2] = x2 * P2[2] - P2[0]
        A[3] = y2 * P2[2] - P2[1]
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        X = X / X[3]
        points.append(X[:3])
    return np.array(points)


def select_R_t(E, K, pts1, pts2, candidates):
    """在 4 组 (R,t) 中选择正深度点最多的解。"""
    P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
    best = None
    best_score = -1
    for R, t in candidates:
        P2 = K @ np.hstack([R, t.reshape(3, 1)])
        X = triangulate_linear(P1, P2, pts1, pts2)
        depth1 = X[:, 2]
        Xc2 = (R @ X.T + t.reshape(3, 1)).T
        depth2 = Xc2[:, 2]
        score = int(np.sum((depth1 > 0) & (depth2 > 0)))
        if score > best_score:
            best_score = score
            best = (R, t, X)
    return best


In [ ]:
# 读取立体图像对
img1 = cv_imread("stereo_image_1.jpg", cv2.IMREAD_GRAYSCALE)
img2 = cv_imread("stereo_image_2.jpg", cv2.IMREAD_GRAYSCALE)
assert img1 is not None and img2 is not None, "读取立体图像对失败"
print(f"图像尺寸: {img1.shape[1]}x{img1.shape[0]}")


In [ ]:
# 特征点提取与匹配（允许调用 OpenCV）
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)
bf = cv2.BFMatcher(cv2.NORM_L2)
knns = bf.knnMatch(des1, des2, k=2)
good = [m for m, n in knns if m.distance < 0.7 * n.distance]
print(f"良好匹配点对数: {len(good)}")

pts1 = np.float64([kp1[m.queryIdx].pt for m in good])
pts2 = np.float64([kp2[m.trainIdx].pt for m in good])


In [ ]:
# 手写 8 点法求 F / E，并与 OpenCV 对比
F = compute_F_8point(pts1, pts2)
F_cv, mask = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC)
print("手写 F:")
print(F)
print("OpenCV F:")
print(F_cv)

K = np.array([[600, 0, 250], [0, 600, 200], [0, 0, 1]], dtype=np.float64)
E = compute_E(K, K, F)
print("本质矩阵 E:")
print(E)


In [ ]:
# E 分解 + cheirality 选择 + 三角化
candidates = decompose_E(E)
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])
R, t, points_3d = select_R_t(E, K, pts1, pts2, candidates)
print("选择得到的旋转矩阵 R:")
print(R)
print("平移向量 t:")
print(t)
print(f"三角化点云数量: {len(points_3d)}")


In [ ]:
# 保存点云到 TXT 文件（CloudCompare / MeshLab 可打开）
out_file = "point_cloud.txt"
with open(out_file, "w", encoding="utf-8") as f:
    f.write("# 3D 点云坐标 (X, Y, Z)\n")
    f.write(f"# 点数: {len(points_3d)}\n")
    f.write("# 格式: X Y Z\n")
    for p in points_3d:
        f.write(f"{p[0]:.6f} {p[1]:.6f} {p[2]:.6f}\n")
print(f"点云已保存: {out_file}")

with open("camera_params.txt", "w", encoding="utf-8") as f:
    f.write("# 相机内参数矩阵 K\n")
    for row in K:
        f.write(f"{row[0]:.6f} {row[1]:.6f} {row[2]:.6f}\n")
    f.write("\n# 相机 2 旋转矩阵 R\n")
    for row in R:
        f.write(f"{row[0]:.6f} {row[1]:.6f} {row[2]:.6f}\n")
    f.write("\n# 相机 2 平移向量 t\n")
    f.write(f"{t[0]:.6f}\n{t[1]:.6f}\n{t[2]:.6f}\n")
print("相机参数已保存: camera_params.txt")


In [ ]:
# 3D 点云多视角可视化
fig = plt.figure(figsize=(14, 5))
views = [(25, 45), (0, 90), (90, 0)]
titles = ["等轴视图", "侧视图", "俯视图"]
for i, (elev, azim) in enumerate(views):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    ax.scatter(points_3d[:, 0], points_3d[:, 1], points_3d[:, 2], c="steelblue", s=12, alpha=0.7)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(titles[i])
    ax.view_init(elev=elev, azim=azim)
plt.suptitle("3D 点云 - 多视角观察")
plt.tight_layout()
plt.show()


## 三、结果与参数分析

- 手写 8 点法 F 与 OpenCV `findFundamentalMat` 求得的 F 方向一致（可能差一个尺度，因为 F 定义到尺度）。
- `E` 分解得到 4 组候选，只有 cheirality 检查选出的解能让大部分点在两个相机中都为正深度。
- 点云质量取决于匹配点数量与内参准确性；这里 `K` 为假设内参，若使用真实标定内参，重建尺度与形状更准。

**易错点**
1. 8 点法必须归一化，否则矩阵病态。
2. 强制 `rank(F)=2` 是把最小奇异值置 0，否则对极约束不严格成立。
3. `R` 需保证行列式为 +1（旋转矩阵），`E` 分解后要检查并修正。


## 四、科研规范小结

1. **可解释的多步管线**：F → E → R/t → 三角化，每步独立函数、可单独验证。
2. **几何约束显式处理**：rank-2 约束、det(R)=+1、正深度选择，体现对几何本质的理解。
3. **输出标准化**：点云保存为 `X Y Z` 文本，兼容 CloudCompare / MeshLab，便于后续处理。


## 五、练习：统计重投影误差

**要求**：把三角化得到的三维点用 `P1/P2` 重新投影回两幅图像，计算平均重投影误差；若误差较大，说明可能的原因。


In [ ]:
# ==================== 练习解决方案 ====================
P2 = K @ np.hstack([R, t.reshape(3, 1)])
Xh = np.hstack([points_3d, np.ones((len(points_3d), 1))])
proj1 = (P1 @ Xh.T).T
proj2 = (P2 @ Xh.T).T
proj1 = proj1[:, :2] / proj1[:, 2:3]
proj2 = proj2[:, :2] / proj2[:, 2:3]
err1 = np.sqrt(np.mean(np.sum((proj1 - pts1) ** 2, axis=1)))
err2 = np.sqrt(np.mean(np.sum((proj2 - pts2) ** 2, axis=1)))
print(f"图1 平均重投影误差: {err1:.4f} px")
print(f"图2 平均重投影误差: {err2:.4f} px")
